# Panel Flow Models

This notebook demonstrates the panel flow model classes and their standard `fit` entry points: the resolvent-gradient sampler for the unrestricted Gaussian model, NUTS for the separable Gaussian model, and the Pólya–Gamma Gibbs sampler (the default) for the NB count models.

The examples use very small synthetic panels so the notebook executes quickly during the docs build while still exercising the full workflow for:

- `SARFlowPanel`
- `SARFlowSeparablePanel`
- `SARNegBinFlowPanel`
- `SARNegBinFlowSeparablePanel`

In [ ]:
import warnings

import numpy as np
import pandas as pd

from bayespecon.dgp.flows import (
    generate_panel_flow_data,
    generate_panel_flow_data_separable,
    generate_panel_negbin_flow_data,
    generate_panel_negbin_flow_data_separable,
)
from bayespecon.models import (
    SARFlowPanel,
    SARFlowSeparablePanel,
    SARNegBinFlowPanel,
    SARNegBinFlowSeparablePanel,
)

warnings.filterwarnings("ignore", category=FutureWarning)
np.set_printoptions(precision=3, suppress=True)
pd.options.display.float_format = "{:.3f}".format

In [ ]:
import arviz as az

In [ ]:
# Each panel DGP shares the auto-G + default `log_distance` behaviour of
# the cross-sectional flow DGPs: omit `G`/`gdf` and a synthetic point grid +
# KNN graph (`knn_k=4`) is built on the fly. The fitted graph is returned
# in the result dict so the model classes can re-use it.
T = 2
FIT_KW = dict(draws=1000, progressbar=True, random_seed=0)

gaussian_data = generate_panel_flow_data(
    n=20,
    T=T,
    rho_d=0.20,
    rho_o=0.15,
    rho_w=0.05,
    beta_d=[1.0],
    beta_o=[0.4],
    sigma=0.5,
    sigma_alpha=0.2,
    seed=10,
)
gaussian_sep_data = generate_panel_flow_data_separable(
    n=20,
    T=T,
    rho_d=0.25,
    rho_o=0.15,
    beta_d=[1.0],
    beta_o=[0.4],
    sigma=0.5,
    sigma_alpha=0.2,
    seed=11,
)
nb_data = generate_panel_negbin_flow_data(
    n=20,
    T=T,
    rho_d=0.20,
    rho_o=0.10,
    rho_w=0.05,
    beta_d=[0.6],
    beta_o=[0.2],
    alpha=5.0,
    seed=12,
    k=1,
)
nb_sep_data = generate_panel_negbin_flow_data_separable(
    n=20,
    T=T,
    rho_d=0.20,
    rho_o=0.10,
    beta_d=[0.6],
    beta_o=[0.2],
    alpha=5.0,
    seed=13,
    k=1,
)

# All four DGPs synthesise the same geometry from `n=4`, so any of the
# returned graphs is fine to share with the model constructors below.
G = gaussian_data["G"]

pd.DataFrame(
    [
        {
            "dataset": "Gaussian unrestricted",
            "y_shape": gaussian_data["y"].shape,
            "X_shape": gaussian_data["X"].shape,
            "features": gaussian_data["col_names"],
        },
        {
            "dataset": "Gaussian separable",
            "y_shape": gaussian_sep_data["y"].shape,
            "X_shape": gaussian_sep_data["X"].shape,
            "features": gaussian_sep_data["col_names"],
        },
        {
            "dataset": "NB unrestricted",
            "y_shape": nb_data["y"].shape,
            "X_shape": nb_data["X"].shape,
            "features": nb_data["col_names"],
        },
        {
            "dataset": "NB separable",
            "y_shape": nb_sep_data["y"].shape,
            "X_shape": nb_sep_data["X"].shape,
            "features": nb_sep_data["col_names"],
        },
    ]
)

> **Asymmetric attributes**: `generate_panel_flow_data` and `generate_panel_negbin_flow_data` now support `beta_d` and `beta_o` with different lengths (`k_d ≠ k_o`). The design matrix layout adjusts automatically. For example:
> ```python
> data = generate_panel_flow_data(n=5, T=3, beta_d=[1.0, -0.5], beta_o=[0.4], ...)
> ```

In [ ]:
gaussian_model = SARFlowPanel(
    y=np.log(gaussian_data["y"]),  # latent SAR scale (lognormal default)
    G=G,
    X=gaussian_data["X"],
    T=T,
    col_names=gaussian_data["col_names"],
    effects=0,
)
gaussian_idata = gaussian_model.fit(**FIT_KW)

print(sorted(gaussian_idata.posterior.data_vars))
gaussian_model.summary(var_names=["rho_d", "rho_o", "rho_w", "sigma"], round_to=3)

In [ ]:
gaussian_sep_model = SARFlowSeparablePanel(
    y=np.log(gaussian_sep_data["y"]),  # latent SAR scale (lognormal default)
    G=G,
    X=gaussian_sep_data["X"],
    T=T,
    col_names=gaussian_sep_data["col_names"],
    effects=0,
)
gaussian_sep_idata = gaussian_sep_model.fit(**FIT_KW)

sep_identity_error = np.max(
    np.abs(
        gaussian_sep_idata.posterior["rho_w"].values
        + gaussian_sep_idata.posterior["rho_d"].values
        * gaussian_sep_idata.posterior["rho_o"].values
    )
)
print(sorted(gaussian_sep_idata.posterior.data_vars))
print(f"max |rho_w + rho_d * rho_o| = {sep_identity_error:.3e}")
gaussian_sep_model.summary(var_names=["rho_d", "rho_o", "rho_w", "sigma"], round_to=3)

## NB panel flow models

The NB panel variants are currently pooled-only (`effects=0`) and default to the reduced-form Pólya–Gamma Gibbs sampler (`sampler="gibbs"`), which keeps the stored posterior compact — no large deterministic `lambda` array is recorded.

In [ ]:
nb_model = SARNegBinFlowPanel(
    y=nb_data["y"],
    G=G,
    X=nb_data["X"],
    T=T,
    col_names=nb_data["col_names"],
    effects=0,
)
nb_idata = nb_model.fit(**FIT_KW)

print(sorted(nb_idata.posterior.data_vars))
print("lambda" in nb_idata.posterior.data_vars)
nb_model.summary(var_names=["rho_d", "rho_o", "rho_w"], round_to=3)

In [ ]:
nb_sep_model = SARNegBinFlowSeparablePanel(
    y=nb_sep_data["y"],
    G=G,
    X=nb_sep_data["X"],
    T=T,
    col_names=nb_sep_data["col_names"],
    effects=0,
)
nb_sep_idata = nb_sep_model.fit(**FIT_KW)

comparison = pd.DataFrame(
    [
        {
            "model": "SARFlowPanel",
            "method": "resolvent-gradient",
            "posterior_vars": ", ".join(sorted(gaussian_idata.posterior.data_vars)),
        },
        {
            "model": "SARFlowSeparablePanel",
            "method": "nuts",
            "posterior_vars": ", ".join(sorted(gaussian_sep_idata.posterior.data_vars)),
        },
        {
            "model": "SARNegBinFlowPanel",
            "method": "gibbs",
            "posterior_vars": ", ".join(sorted(nb_idata.posterior.data_vars)),
        },
        {
            "model": "SARNegBinFlowSeparablePanel",
            "method": "gibbs",
            "posterior_vars": ", ".join(sorted(nb_sep_idata.posterior.data_vars)),
        },
    ]
)

print(sorted(nb_sep_idata.posterior.data_vars))
nb_sep_model.summary(var_names=["rho_d", "rho_o", "rho_w"], round_to=3)

comparison

In [ ]:
az.plot_trace(nb_sep_idata)